# LeetCode 1942 — The Number of the Smallest Unoccupied Chair bằng Python

Link đề: https://leetcode.com/problems/the-number-of-the-smallest-unoccupied-chair/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def smallestChair(self, times: List[List[int]], targetFriend: int) -> int:
        ...
```

Lộ trình: **mô phỏng theo từng đơn vị thời gian, quét ghế từ 0** → **mô phỏng theo sự kiện, tìm ghế trống bằng quét mảng** `O(n²)` → **hai heap: ghế trống (min-heap số ghế) + đang ngồi (min-heap giờ về)** `O(n log n)` (lời giải nộp) → **heap ghế trống + sự kiện đã sắp xếp** (biến thể không cần heap thứ hai).

## 1. Mục tiêu học

1. Mô phỏng hệ thống "đến/đi" bằng **sự kiện theo thời gian**, xử lý đúng **thứ tự khi trùng thời điểm** (người về trước, người đến sau — ghế vừa trống dùng được ngay).
2. Dùng **min-heap ghế trống** để luôn lấy ghế có **số nhỏ nhất** trong `O(log n)`.
3. Dùng **min-heap theo giờ về** để giải phóng ghế đúng lúc.
4. Nhận ra chỉ cần tối đa `n` ghế (`0..n−1`) và có thể **dừng sớm** khi tới người cần tìm.
5. Kiểm chứng bằng mô phỏng thẳng thắn trên input ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Có `n` người bạn đánh số `0..n−1` tới dự tiệc. Có vô hạn ghế đánh số `0, 1, 2, …`. Khi một người tới, họ ngồi vào ghế **trống có số nhỏ nhất** (ví dụ ghế 0, 1, 5 đang có người thì người mới ngồi ghế 2).

Khi một người rời đi, ghế của họ **trống ngay tại thời điểm đó**; nếu có người tới **đúng thời điểm** ấy thì có thể ngồi ghế vừa trống.

Cho mảng `times`, `times[i] = [arrival_i, leaving_i]` là giờ đến và giờ về của người `i`, và số nguyên `targetFriend`. **Mọi giờ đến khác nhau.** Trả về số ghế mà người `targetFriend` ngồi.

**Ví dụ 1:** `times = [[1,4],[2,3],[4,6]], targetFriend = 1` → `1`.

**Ví dụ 2:** `times = [[3,10],[1,5],[2,6]], targetFriend = 0` → `2`.

**Ràng buộc:** `2 ≤ n ≤ 10⁴`, `1 ≤ arrival_i < leaving_i ≤ 10⁵`, `0 ≤ targetFriend ≤ n − 1`, giờ đến đôi một khác nhau.

In [1]:
import heapq
import random
import time
from typing import List

examples = [([[1, 4], [2, 3], [4, 6]], 1, 1), ([[3, 10], [1, 5], [2, 6]], 0, 2)]
for t, f, want in examples:
    print(t, "target =", f, "-> ghế", want)

[[1, 4], [2, 3], [4, 6]] target = 1 -> ghế 1
[[3, 10], [1, 5], [2, 6]] target = 0 -> ghế 2


## 3. Kiến thức nền

### 3.1 `heapq` — min-heap

`heappush`, `heappop`: `O(log n)`; `h[0]` là phần tử nhỏ nhất. Tuple so sánh theo phần tử đầu, rồi phần tử sau.

In [2]:
h = []
for x in [5, 1, 4]:
    heapq.heappush(h, x)
print(h[0], heapq.heappop(h), h)

1 1 [4, 5]


### 3.2 `heapify` một dãy có sẵn trong `O(n)`

Ghế trống ban đầu `0..n−1` đã là heap hợp lệ (list tăng dần) — không cần `heapify`, nhưng gọi cũng không sao.

In [3]:
free = list(range(5))
heapq.heapify(free)
print(free, heapq.heappop(free))

[1, 3, 2, 4] 0


### 3.3 Sắp xếp chỉ số theo giờ đến

In [4]:
times = [[3, 10], [1, 5], [2, 6]]
order = sorted(range(len(times)), key=lambda i: times[i][0])
print("thứ tự đến:", order)

thứ tự đến: [1, 2, 0]


### 3.4 Thứ tự xử lý sự kiện trùng thời điểm

Tại cùng thời điểm `t`: **mọi** người có giờ về `≤ t` phải trả ghế **trước** khi người đến lúc `t` chọn ghế. Điều kiện giải phóng là `leave ≤ arrival` (không phải `<`).

In [5]:
leave, arrive = 4, 4
print("ghế của người về lúc 4 dùng được cho người đến lúc 4?", leave <= arrive)

ghế của người về lúc 4 dùng được cho người đến lúc 4? True


## 4. Ý tưởng nền tảng / trực giác

Mô phỏng Ví dụ 2 theo thứ tự đến (`1, 2, 3`):

| thời điểm | sự kiện | ghế trống | ghế đang dùng | ghi chú |
|---|---|---|---|---|
| 1 | người 1 đến | 0,1,2,… | — | ngồi ghế **0** |
| 2 | người 2 đến | 1,2,… | 0 (về lúc 5) | ngồi ghế **1** |
| 3 | người 0 đến | 2,… | 0 (5), 1 (6) | ngồi ghế **2** ⇒ đáp án |

Ví dụ 1: lúc 4, người 0 **về** (ghế 0 trống) **đúng lúc** người 2 **đến** ⇒ người 2 ngồi ghế 0. Người cần tìm (người 1) đã ngồi ghế 1 lúc 2.

Hai câu hỏi lặp lại liên tục: "ghế trống **nhỏ nhất** là ghế nào?" và "**ai** về sớm nhất (để trả ghế)?" ⇒ hai heap.

## 5. Cách 1 — Brute Force / baseline: mô phỏng từng đơn vị thời gian

**Ý tưởng:** duyệt `t = 1, 2, …, max_leave`. Ở mỗi thời điểm: trước hết cho những người về lúc `t` rời ghế; rồi người đến lúc `t` (nếu có) quét ghế từ 0 tìm ghế trống đầu tiên. Đúng theo câu chữ ⇒ chuẩn đối chiếu.

In [6]:
def smallest_chair_brute(times, target):
    n = len(times)
    arrive_at = {a: i for i, (a, _) in enumerate(times)}
    leave_at = {}
    for i, (_, l) in enumerate(times):
        leave_at.setdefault(l, []).append(i)
    seat = {}                       # người -> ghế
    occupied = [False] * n          # tối đa n ghế được dùng đồng thời
    for t in range(1, max(l for _, l in times) + 1):
        for i in leave_at.get(t, []):
            occupied[seat[i]] = False
        if t in arrive_at:
            i = arrive_at[t]
            c = 0
            while occupied[c]:
                c += 1
            occupied[c] = True; seat[i] = c
            if i == target:
                return c
    return -1

for t, f, want in examples:
    print(smallest_chair_brute(t, f), "| kỳ vọng", want)

1 | kỳ vọng 1
2 | kỳ vọng 2


**Độ phức tạp:** `O(T + n²)` với `T = max leaving ≤ 10⁵` (duyệt mọi thời điểm) và quét ghế `O(n)` mỗi người. Chạy được với ràng buộc, nhưng quét ghế là điểm nghẽn khi `n` lớn và lặp lại mọi thời điểm "rỗng" là lãng phí.

**Nút thắt:** (1) tìm ghế trống nhỏ nhất bằng quét; (2) duyệt cả các thời điểm không có sự kiện.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (chỉ xét thời điểm đến):** sắp xếp người theo giờ đến. Trước khi xếp ghế cho người đến lúc `a`, giải phóng **mọi** ghế có giờ về `≤ a`.

**Insight 2 (hai heap):**
- `free`: min-heap các **số ghế** đang trống — `heappop` cho ghế nhỏ nhất.
- `busy`: min-heap `(giờ về, số ghế)` của người đang ngồi — trong khi `busy[0][0] ≤ a`: pop và trả ghế vào `free`.

**Insight 3 (đủ `n` ghế):** không bao giờ có quá `n` người ngồi cùng lúc ⇒ khởi tạo `free = [0..n−1]` là đủ.

**Quy tắc:** với từng người theo thứ tự đến: giải phóng ghế của những người về `≤ a` → lấy ghế nhỏ nhất → nếu là `targetFriend` thì trả về ngay.

**Invariant:** trước mỗi lần xếp ghế, `free` chứa đúng các ghế `0..n−1` không bị ai đang ngồi tại thời điểm `a` (sau khi đã trả ghế những người về `≤ a`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng theo sự kiện, tìm ghế bằng quét (`O(n²)`)

In [7]:
def smallest_chair_scan(times, target):
    n = len(times)
    order = sorted(range(n), key=lambda i: times[i][0])
    free_at = [0] * n                    # free_at[c] = thời điểm ghế c trống trở lại (0 = trống từ đầu)
    for i in order:
        a, l = times[i]
        c = 0
        while free_at[c] > a:            # ghế c còn người (giờ về > a)
            c += 1
        free_at[c] = l
        if i == target:
            return c
    return -1

for t, f, want in examples:
    print(smallest_chair_scan(t, f), want)

1 1
2 2


Mẹo hay: lưu `free_at[c]` (giờ ghế `c` trống trở lại) thay vì danh sách người — ghế `c` dùng được cho người đến lúc `a` ⇔ `free_at[c] ≤ a`. **Độ phức tạp:** sắp xếp `O(n log n)` + quét `O(n)` mỗi người ⇒ `O(n²)` worst = `10⁸` với `n = 10⁴` — sát giới hạn trong Python.

### 7.2 Cách 3 — Hai heap (`O(n log n)`) — lời giải chuẩn

In [8]:
def smallest_chair_heaps_demo(times, target):
    n = len(times)
    order = sorted(range(n), key=lambda i: times[i][0])
    free = list(range(n))
    busy = []                                       # (giờ về, ghế)
    for i in order:
        a, l = times[i]
        while busy and busy[0][0] <= a:
            lt, c = heapq.heappop(busy)
            heapq.heappush(free, c)
            print(f"  t={a}: ghế {c} trống lại (người về lúc {lt})")
        c = heapq.heappop(free)
        print(f"t={a}: người {i} ngồi ghế {c} (về lúc {l})")
        if i == target:
            return c
        heapq.heappush(busy, (l, c))
    return -1

print("=>", smallest_chair_heaps_demo(*examples[0][:2]))

t=1: người 0 ngồi ghế 0 (về lúc 4)
t=2: người 1 ngồi ghế 1 (về lúc 3)
=> 1


In [9]:
class SolutionHeaps:
    def smallestChair(self, times: List[List[int]], targetFriend: int) -> int:
        n = len(times)
        order = sorted(range(n), key=lambda i: times[i][0])
        free = list(range(n))
        busy = []
        for i in order:
            a, l = times[i]
            while busy and busy[0][0] <= a:
                heapq.heappush(free, heapq.heappop(busy)[1])
            c = heapq.heappop(free)
            if i == targetFriend:
                return c
            heapq.heappush(busy, (l, c))
        return -1

for t, f, want in examples:
    print(SolutionHeaps().smallestChair(t, f), want)

1 1
2 2


**Độ phức tạp:** sắp xếp `O(n log n)`; mỗi người một lần push/pop ở mỗi heap ⇒ `O(n log n)`; bộ nhớ `O(n)`.

### 7.3 Cách 4 — Heap ghế trống + danh sách sự kiện "về" đã sắp xếp

Thay heap `busy` bằng cách **sắp xếp trước** tất cả sự kiện về; dùng con trỏ đi theo. Nhưng ta chưa biết ghế của người về khi sắp xếp — nên lưu `seat[i]` khi xếp ghế, và chỉ giải phóng những người **đã** ngồi (mọi người về `≤ a` đều đã đến trước đó vì `arrival < leaving`).

In [10]:
class SolutionEvents:
    def smallestChair(self, times: List[List[int]], targetFriend: int) -> int:
        n = len(times)
        arrivals = sorted(range(n), key=lambda i: times[i][0])
        leaves = sorted(range(n), key=lambda i: times[i][1])
        seat = [-1] * n
        free = list(range(n))
        p = 0
        for i in arrivals:
            a = times[i][0]
            while p < n and times[leaves[p]][1] <= a:
                heapq.heappush(free, seat[leaves[p]])     # người này chắc chắn đã ngồi (đến < về <= a)
                p += 1
            seat[i] = heapq.heappop(free)
            if i == targetFriend:
                return seat[i]
        return -1

for t, f, want in examples:
    print(SolutionEvents().smallestChair(t, f), want)

1 1
2 2


**Độ phức tạp:** hai lần sắp xếp `O(n log n)` + heap `O(n log n)`. Ý tưởng "sắp xếp sự kiện + con trỏ" thay cho heap thứ hai khi các sự kiện đã biết trước toàn bộ (offline).

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 1 (`target = 1`):

| người (theo giờ đến) | `a` | giải phóng (`busy[0][0] ≤ a`) | `free` trước khi chọn | ghế | `busy` sau |
|---|---|---|---|---|---|
| 0 | 1 | — | `[0,1,2]` | 0 | `[(4,0)]` |
| 1 | 2 | — | `[1,2]` | **1** ⇒ trả về | |

Nếu target là 2: tiếp tục — lúc `a = 4`: `(3,1)` và `(4,0)` đều `≤ 4` ⇒ trả ghế 1, 0; `free = [0,1,2]` ⇒ người 2 ngồi ghế 0.

In [11]:
def trace_heaps(times, target):
    n = len(times); order = sorted(range(n), key=lambda i: times[i][0])
    free = list(range(n)); busy = []
    for i in order:
        a, l = times[i]
        released = []
        while busy and busy[0][0] <= a:
            lt, c = heapq.heappop(busy); heapq.heappush(free, c); released.append((lt, c))
        c = heapq.heappop(free)
        print(f"người {i} đến lúc {a}: trả ghế {released} -> free trước khi chọn {sorted(free + [c])} -> ngồi {c}")
        if i == target:
            return c
        heapq.heappush(busy, (l, c))

trace_heaps([[1, 4], [2, 3], [4, 6]], 2)

người 0 đến lúc 1: trả ghế [] -> free trước khi chọn [0, 1, 2] -> ngồi 0
người 1 đến lúc 2: trả ghế [] -> free trước khi chọn [1, 2] -> ngồi 1
người 2 đến lúc 4: trả ghế [(3, 1), (4, 0)] -> free trước khi chọn [0, 1, 2] -> ngồi 0


0

In [12]:
trace_heaps(*examples[1][:2])

người 1 đến lúc 1: trả ghế [] -> free trước khi chọn [0, 1, 2] -> ngồi 0
người 2 đến lúc 2: trả ghế [] -> free trước khi chọn [1, 2] -> ngồi 1
người 0 đến lúc 3: trả ghế [] -> free trước khi chọn [2] -> ngồi 2


2

## 9. Code nộp LeetCode

In [13]:
import heapq
from typing import List

class Solution:
    def smallestChair(self, times: List[List[int]], targetFriend: int) -> int:
        n = len(times)
        order = sorted(range(n), key=lambda i: times[i][0])   # xử lý theo giờ đến
        free = list(range(n))                                # min-heap ghế trống (đã là heap)
        busy = []                                            # min-heap (giờ về, ghế)
        for i in order:
            arrive, leave = times[i]
            while busy and busy[0][0] <= arrive:             # về trước hoặc ĐÚNG lúc này: trả ghế
                heapq.heappush(free, heapq.heappop(busy)[1])
            chair = heapq.heappop(free)                      # ghế trống nhỏ nhất
            if i == targetFriend:
                return chair
            heapq.heappush(busy, (leave, chair))
        return -1

for t, f, want in examples:
    assert Solution().smallestChair(t, f) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Chỉ cần xét các thời điểm đến.** Việc chọn ghế chỉ xảy ra khi có người đến; trạng thái ghế giữa hai lần đến chỉ thay đổi do người về — mà ta xử lý gộp ngay trước lần đến kế tiếp.

**Bất biến:** ngay trước khi xếp ghế cho người đến lúc `a`, `free` = tập ghế trong `0..n−1` **không** có người ngồi tại thời điểm `a`. *Chứng minh:* ghế của người `j` bị giữ khỏi `free` từ lúc `j` ngồi tới khi ta pop `j` khỏi `busy`; ta pop đúng khi `leave_j ≤ a` (luật "trống ngay tại thời điểm rời đi" — kể cả khi trùng giờ đến). Không ai rời trước khi đến (`arrival < leaving`).

**Đủ ghế:** tại mọi thời điểm có tối đa `n` người ⇒ ghế nhỏ nhất còn trống luôn `< n` ⇒ chỉ cần `0..n−1`.

**Chọn đúng ghế:** `heappop(free)` trả ghế trống **nhỏ nhất** — đúng luật. Mọi giờ đến khác nhau nên không có tranh chấp giữa hai người đến cùng lúc.

## 11. Phân tích Time / Space

Gọi `n ≤ 10⁴`, `T = max leaving ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Theo từng thời điểm | `O(T + n²)` | `O(n + T)` |
| 2. Sự kiện + quét ghế | `O(n log n + n²)` | `O(n)` |
| 3. Hai heap | `O(n log n)` | `O(n)` |
| 4. Heap + sự kiện về sắp xếp | `O(n log n)` | `O(n)` |

In [14]:
def gen_times(n, rng, T=10**5):
    arr = rng.sample(range(1, T), n)
    return [[a, rng.randint(a + 1, T)] for a in arr]

rng = random.Random(1942)
times = gen_times(10**4, rng)
target = max(range(len(times)), key=lambda i: times[i][0])           # người đến muộn nhất: mô phỏng hết
nested = [[i + 1, 2 * 10**4 + 1 - i] for i in range(10**4)]          # lồng nhau: ai cũng còn ngồi -> quét O(n) mỗi lần
for label, (ts, tg) in [("ngẫu nhiên", (times, target)), ("lồng nhau", (nested, 10**4 - 1))]:
    res = []
    for name, f in [("quét ghế", smallest_chair_scan), ("hai heap/Nộp", Solution().smallestChair),
                    ("heap + sự kiện", SolutionEvents().smallestChair)]:
        t0 = time.perf_counter(); r = f(ts, tg); res.append(r)
        print(f"[{label}] {name}: ghế {r} trong {1000*(time.perf_counter()-t0):.0f} ms")
    assert len(set(res)) == 1

[ngẫu nhiên] quét ghế: ghế 1 trong 338 ms
[ngẫu nhiên] hai heap/Nộp: ghế 1 trong 12 ms
[ngẫu nhiên] heap + sự kiện: ghế 1 trong 9 ms


[lồng nhau] quét ghế: ghế 9999 trong 1169 ms
[lồng nhau] hai heap/Nộp: ghế 9999 trong 9 ms
[lồng nhau] heap + sự kiện: ghế 9999 trong 4 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu mô phỏng từng thời điểm)

In [15]:
sols = [Solution(), SolutionHeaps(), SolutionEvents()]
edge = examples + [
    ([[1, 2], [2, 3]], 1, 0),          # về đúng lúc người kia đến: dùng lại ghế 0
    ([[1, 3], [2, 3]], 1, 1),
    ([[1, 5], [2, 3], [3, 4]], 2, 1),  # người 1 về lúc 3, người 2 đến lúc 3 -> ghế 1
    ([[5, 6], [1, 10], [2, 3]], 0, 1), # người 2 (ghế 1) về lúc 3, người 0 đến lúc 5 -> ghế 1
    ([[1, 100], [2, 100], [3, 100], [4, 100]], 3, 3),
]
for t, f, want in edge:
    assert smallest_chair_brute(t, f) == want, (t, f, smallest_chair_brute(t, f))
    for s in sols:
        assert s.smallestChair(t, f) == want, (t, f, s)
    assert smallest_chair_scan(t, f) == want
    print("OK", t, "target =", f, "->", want)

OK [[1, 4], [2, 3], [4, 6]] target = 1 -> 1
OK [[3, 10], [1, 5], [2, 6]] target = 0 -> 2
OK [[1, 2], [2, 3]] target = 1 -> 0
OK [[1, 3], [2, 3]] target = 1 -> 1
OK [[1, 5], [2, 3], [3, 4]] target = 2 -> 1
OK [[5, 6], [1, 10], [2, 3]] target = 0 -> 1
OK [[1, 100], [2, 100], [3, 100], [4, 100]] target = 3 -> 3


### 12.2 Ngẫu nhiên có seed: mọi người cần tìm trên dữ liệu nhỏ

In [16]:
rng = random.Random(42)
for _ in range(400):
    n = rng.randint(2, 10)
    T = rng.choice([15, 30, 60])
    ts = gen_times(n, rng, T)
    for tg in range(n):
        want = smallest_chair_brute(ts, tg)
        for s in sols:
            assert s.smallestChair(ts, tg) == want, (ts, tg)
        assert smallest_chair_scan(ts, tg) == want
print("400 bộ × mọi targetFriend: khớp mô phỏng từng thời điểm")

400 bộ × mọi targetFriend: khớp mô phỏng từng thời điểm


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Giải phóng ghế khi `leave < arrive` (bỏ trường hợp bằng nhau)

In [17]:
def wrong_strict(times, target):
    n = len(times); order = sorted(range(n), key=lambda i: times[i][0])
    free = list(range(n)); busy = []
    for i in order:
        a, l = times[i]
        while busy and busy[0][0] < a:                          # SAI: phải là <=
            heapq.heappush(free, heapq.heappop(busy)[1])
        c = heapq.heappop(free)
        if i == target: return c
        heapq.heappush(busy, (l, c))

t, f = [[1, 4], [2, 3], [4, 6]], 2
print("sai:", wrong_strict(t, f), "| đúng:", Solution().smallestChair(t, f))

sai: 1 | đúng: 0


**Vì sao sai:** đề nói ghế trống **ngay lúc** người rời đi và người đến cùng lúc **được** ngồi. Người 0 về lúc 4, người 2 đến lúc 4 ⇒ ghế 0 phải dùng được.

### Lỗi 2 — Cấp ghế mới bằng số người đang ngồi (`len(busy)`)

In [18]:
def wrong_count(times, target):
    n = len(times); order = sorted(range(n), key=lambda i: times[i][0])
    busy = []
    for i in order:
        a, l = times[i]
        while busy and busy[0][0] <= a:
            heapq.heappop(busy)
        c = len(busy)                                            # SAI: ghế nhỏ nhất chưa chắc là len(busy)
        if i == target: return c
        heapq.heappush(busy, (l, c))

t, f = [[7, 8], [2, 5], [3, 9]], 0
print("sai:", wrong_count(t, f), "| đúng:", Solution().smallestChair(t, f))

sai: 1 | đúng: 0


**Vì sao sai:** người 1 (đến lúc 2) ngồi ghế 0, người 2 (đến lúc 3) ngồi ghế 1. Lúc 7, người 1 đã về (lúc 5) nên ghế **0** trống, còn ghế 1 vẫn có người. `len(busy) = 1` lại trỏ vào ghế 1 — **đang có người**. Số người đang ngồi không cho biết **ghế nào** trống khi người ở ghế thấp rời đi trước. (Phản ví dụ tìm bằng tìm kiếm ngẫu nhiên, đối chiếu mô phỏng.) **Sửa:** heap ghế trống.

### Lỗi 3 — Xử lý theo chỉ số người thay vì theo giờ đến

In [19]:
def wrong_order(times, target):
    n = len(times); free = list(range(n)); busy = []
    for i in range(n):                                           # SAI: chưa sắp xếp theo giờ đến
        a, l = times[i]
        while busy and busy[0][0] <= a:
            heapq.heappush(free, heapq.heappop(busy)[1])
        c = heapq.heappop(free)
        if i == target: return c
        heapq.heappush(busy, (l, c))

t, f, want = examples[1]
print("sai:", wrong_order(t, f), "| đúng:", want)

sai: 0 | đúng: 2


**Vì sao sai:** người 0 đến lúc 3 nhưng được xử lý trước người 1 (đến lúc 1) ⇒ tưởng ghế 0 trống. **Sửa:** sắp xếp theo `arrival`.

### Lỗi 4 — Chỉ giải phóng **một** người mỗi lần (dùng `if` thay vì `while`)

In [20]:
def wrong_if(times, target):
    n = len(times); order = sorted(range(n), key=lambda i: times[i][0])
    free = list(range(n)); busy = []
    for i in order:
        a, l = times[i]
        if busy and busy[0][0] <= a:                             # SAI: có thể nhiều người đã về
            heapq.heappush(free, heapq.heappop(busy)[1])
        c = heapq.heappop(free)
        if i == target: return c
        heapq.heappush(busy, (l, c))

t, f = [[1, 4], [2, 3], [4, 6]], 2
print("sai:", wrong_if(t, f), "| đúng:", Solution().smallestChair(t, f))

sai: 1 | đúng: 0


**Vì sao sai:** lúc 4, **hai** người (về lúc 3 và lúc 4) đều đã trả ghế. `if` chỉ trả một ghế (ghế 1) nên người đến lúc 4 ngồi ghế 1 thay vì ghế 0.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tài nguyên đánh số, luôn cấp số nhỏ nhất còn trống, có giải phóng theo thời gian"** ⇒ **heap tài nguyên trống + heap theo thời điểm giải phóng** (LeetCode 1882 *Process Tasks Using Servers*, 2402 *Meeting Rooms III*).
- Sự kiện trùng thời điểm: xác định rõ thứ tự (ở đây: rời đi trước, đến sau) ⇒ `<=`.
- Giải phóng bằng `while`, không phải `if`.
- Số tài nguyên tối đa cần dùng ≤ số người ⇒ khởi tạo heap `0..n−1`.
- Dừng sớm khi tới đối tượng cần hỏi.

## 15. Biến thể của bài

1. **Trả về ghế của mọi người** (Bài tập 3).
2. **Số ghế tối thiểu cần chuẩn bị** = số người có mặt đồng thời nhiều nhất (sweep line) — LeetCode 253 *Meeting Rooms II*.
3. **LeetCode 2402 — Meeting Rooms III:** phòng bận thì cuộc họp **dời** tới khi phòng sớm nhất trống.
4. **LeetCode 1882 — Process Tasks Using Servers:** server trống ưu tiên theo `(trọng số, chỉ số)`.
5. **Người ngồi ghế lớn nhất / gần cửa nhất** ⇒ đổi khóa của heap.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Theo thời điểm | mô phỏng thẳng | `O(T + n²)` | `O(n + T)` | kiểm chứng |
| 2. Sự kiện + quét | `free_at[c] ≤ a` | `O(n²)` | `O(n)` | `n` nhỏ, code ngắn |
| 3. Hai heap | ghế trống + giờ về | `O(n log n)` | `O(n)` | **nộp bài** |
| 4. Heap + sự kiện | sắp xếp giờ về + con trỏ | `O(n log n)` | `O(n)` | offline, tránh heap thứ hai |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `([[1,2],[2,3]], 1)`, `([[1,5],[2,6],[3,7]], 2)`?

**Bài 2 (tính tay):** Lập bảng `free`/`busy` cho `times = [[1,10],[2,3],[4,6],[5,7]]` tới người 3.

**Bài 3 (điền TODO):** Viết `all_chairs(times)` trả về list ghế của **mọi** người (theo chỉ số người).

**Bài 4 (tự viết lại):** Viết `min_chairs_needed(times)` — số ghế tối thiểu để mọi người đều có chỗ (với luật "về rồi người khác ngồi ngay được").

**Bài 5 (mini challenge):** Biến thể: ghế bị **bẩn** sau khi có người rời, cần `c` đơn vị thời gian dọn mới dùng lại được (ghế trống từ thời điểm `leave + c`). Viết `smallest_chair_cleaning(times, target, c)` và đối chiếu mô phỏng.

In [21]:
def all_chairs(times):
    # TODO
    pass

def min_chairs_needed(times):
    # TODO
    pass

def smallest_chair_cleaning(times, target, c):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Mô phỏng tay | Câu 1: ghế 0 trống lại lúc 2 ⇒ 0 | Câu 2: ba người chồng nhau ⇒ 2 |
| 2 | Lúc 1: người 0 → 0 | Lúc 2: người 1 → 1; lúc 4: ghế 1 trống (về 3) → người 2 → 1 | Lúc 5: ghế 2 |
| 3 | Bỏ `return` sớm | Ghi `seat[i]` | Trả về `seat` |
| 4 | Đây là số người có mặt đồng thời nhiều nhất | Sự kiện: về `(l, −1)` trước đến `(a, +1)` khi trùng | Hoặc chạy `all_chairs` rồi `max + 1` |
| 5 | Đưa vào `busy` với khóa `leave + c` | Giải phóng khi `leave + c ≤ arrive` | Mô phỏng: `free_at[c] = leave + c` |

## 19. Đáp án tham khảo

In [22]:
r1 = [Solution().smallestChair(*a) for a in [([[1, 2], [2, 3]], 1), ([[1, 5], [2, 6], [3, 7]], 2)]]
print("Bài 1:", r1); assert r1 == [0, 2]
print("Bài 2:"); assert trace_heaps([[1, 10], [2, 3], [4, 6], [5, 7]], 3) == Solution().smallestChair([[1, 10], [2, 3], [4, 6], [5, 7]], 3)

Bài 1: [0, 2]
Bài 2:
người 0 đến lúc 1: trả ghế [] -> free trước khi chọn [0, 1, 2, 3] -> ngồi 0
người 1 đến lúc 2: trả ghế [] -> free trước khi chọn [1, 2, 3] -> ngồi 1
người 2 đến lúc 4: trả ghế [(3, 1)] -> free trước khi chọn [1, 2, 3] -> ngồi 1
người 3 đến lúc 5: trả ghế [] -> free trước khi chọn [2, 3] -> ngồi 2


In [23]:
def all_chairs(times):
    n = len(times); order = sorted(range(n), key=lambda i: times[i][0])
    free = list(range(n)); busy = []; seat = [-1] * n
    for i in order:
        a, l = times[i]
        while busy and busy[0][0] <= a:
            heapq.heappush(free, heapq.heappop(busy)[1])
        seat[i] = heapq.heappop(free)
        heapq.heappush(busy, (l, seat[i]))
    return seat

def min_chairs_needed(times):
    events = sorted([(l, -1) for _, l in times] + [(a, 1) for a, _ in times])
    cur = best = 0
    for _, d in events:
        cur += d; best = max(best, cur)
    return best

rng = random.Random(3)
for _ in range(300):
    ts = gen_times(rng.randint(2, 12), rng, 40)
    seats = all_chairs(ts)
    assert seats == [smallest_chair_brute(ts, i) for i in range(len(ts))]
    assert min_chairs_needed(ts) == max(seats) + 1
print("Bài 3, 4: OK", all_chairs(examples[1][0]), min_chairs_needed(examples[1][0]))

Bài 3, 4: OK [2, 0, 1] 3


In [24]:
def smallest_chair_cleaning(times, target, c):
    n = len(times); order = sorted(range(n), key=lambda i: times[i][0])
    free = list(range(n)); busy = []
    for i in order:
        a, l = times[i]
        while busy and busy[0][0] <= a:
            heapq.heappush(free, heapq.heappop(busy)[1])
        ch = heapq.heappop(free)
        if i == target: return ch
        heapq.heappush(busy, (l + c, ch))

def cleaning_sim(times, target, c):
    n = len(times); order = sorted(range(n), key=lambda i: times[i][0])
    free_at = [0] * n
    for i in order:
        a, l = times[i]
        ch = next(k for k in range(n) if free_at[k] <= a)
        free_at[ch] = l + c
        if i == target: return ch

rng = random.Random(9)
for _ in range(400):
    ts = gen_times(rng.randint(2, 10), rng, 40); c = rng.randint(0, 5)
    for tg in range(len(ts)):
        assert smallest_chair_cleaning(ts, tg, c) == cleaning_sim(ts, tg, c)
    assert all(smallest_chair_cleaning(ts, tg, 0) == Solution().smallestChair(ts, tg) for tg in range(len(ts)))
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", smallest_chair_cleaning([[1, 4], [2, 3], [4, 6]], 2, 1))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
order = sorted(range(n), key=lambda i: times[i][0])
free, busy = list(range(n)), []           # ghế trống (min-heap) | (giờ về, ghế)
for i in order:
    a, l = times[i]
    while busy and busy[0][0] <= a:        # <= : về đúng lúc cũng trả ghế
        heappush(free, heappop(busy)[1])
    c = heappop(free)
    if i == target: return c
    heappush(busy, (l, c))
```

**Checklist:**
- [ ] Xử lý theo **giờ đến** (sắp xếp)?
- [ ] Giải phóng bằng `while` với điều kiện `<=`?
- [ ] Ghế trống nhỏ nhất lấy từ **heap**, không phải `len(busy)`?
- [ ] Khởi tạo đủ ghế `0..n−1`?
- [ ] Dừng sớm khi tới `targetFriend`.